In [339]:
def build_fourfacts(year : int) -> pd.DataFrame:
    data = sdv.mbb.load_mbb_team_boxscore(seasons=[year], return_as_pandas = True)
    pbp = sdv.mbb.load_mbb_pbp(seasons=[year], return_as_pandas = True)
    pbp = pbp.dropna(subset = ['team_id'])
    pbp['opponent_team_id'] = np.where((pbp['team_id'] == pbp['home_team_id']), pbp['away_team_id'], pbp['home_team_id'])

    end_poss = ['Defensive Rebound', 'Lost Ball Turnover', 'End Period', 'RegularTimeOut', 'End Game', 'Technical Foul', 'MadeFreeThrow']

    pbp['2nd_ft'] = (pbp['type_text'] == 'MadeFreeThrow') & (pbp['type_text'].shift(1) == 'MadeFreeThrow')

    pbp['poss_change'] = ((pbp['type_text'].isin(end_poss) | (pbp['scoring_play'])) | (pbp['type_text'] == 'Steal') & (pbp['type_text'].shift(1) != 'Lost Ball Turnover')) & (~pbp['2nd_ft'])
    pbp['new_poss'] = (pbp['poss_change'].shift(1))

    pbp['rim_fga'] = (pbp['type_text'].isin(['LayUpShot', 'DunkShot', 'TipShot'])) & (pbp['score_value'] == 2)
    pbp['rim_fgm'] = pbp['rim_fga'] & (pbp['scoring_play'])
    pbp['mid_fga'] = (pbp['type_text'].isin(['JumpShot',])) & (pbp['score_value'] == 2)
    pbp['mid_fgm'] = pbp['mid_fga'] & (pbp['scoring_play'])



    off_poss_count = pbp.groupby(['game_id', 'team_id', 'opponent_team_id'])[['new_poss', 'rim_fga', 'rim_fgm', 'mid_fga', 'mid_fgm']].sum().reset_index()
    off_poss_count = pd.merge(off_poss_count, off_poss_count, left_on = ['game_id', 'opponent_team_id'], right_on = ['game_id', 'team_id'], suffixes = ('', '_opp'))
    off_poss_count = off_poss_count.groupby('team_id')[['new_poss', 'rim_fga', 'rim_fgm', 'mid_fga', 'mid_fgm', 'new_poss_opp', 'rim_fga_opp', 'rim_fgm_opp', 'mid_fga_opp', 'mid_fgm_opp']].sum().reset_index()

    pbp['time_diff'] =  pd.to_datetime(pbp['wallclock']) - pd.to_datetime(pbp['wallclock'].shift(1))
    pbp['time_diff'] = pbp['time_diff'].dt.total_seconds()
    pbp['transition'] = (pbp['time_diff'] <= 8) & (pbp['time_diff'] >= 0) & (pbp['new_poss']) & (pbp['score_value'] > 1)

    transitions = pbp[pbp['transition']].copy()
    transitions['3pa'] = transitions['score_value'] == 3
    transitions['2pa'] = transitions['score_value'] == 2
    transitions['3pm'] = transitions['3pa'] & transitions['scoring_play']
    transitions['2pm'] = transitions['2pa'] & transitions['scoring_play']

    transition_stats = transitions.groupby(['game_id', 'team_id', 'opponent_team_id'])[['3pa', '3pm', '2pa', '2pm']].sum()
    transition_stats = transition_stats.add_suffix("_transition")
    transition_stats = transition_stats.reset_index()
    transition_stats = pd.merge(transition_stats, transition_stats, left_on=['game_id', 'opponent_team_id'], right_on=['game_id', 'team_id'], suffixes=('', '_opp'))
    transition_stats = transition_stats.groupby('team_id')[['3pa_transition', '3pm_transition', '2pa_transition', '2pm_transition',
                                        '3pa_transition_opp', '3pm_transition_opp', '2pa_transition_opp', '2pm_transition_opp']].sum().reset_index()

    m_data = data[['game_id', 'team_id', 'assists', 'blocks', 'defensive_rebounds',
       'fast_break_points', 'field_goal_pct', 'field_goals_made',
       'field_goals_attempted', 'flagrant_fouls', 'fouls', 'free_throw_pct',
       'free_throws_made', 'free_throws_attempted', 'largest_lead',
       'lead_changes', 'lead_percentage', 'offensive_rebounds',
       'points_in_paint', 'steals', 'team_turnovers', 'technical_fouls',
       'three_point_field_goal_pct', 'three_point_field_goals_made',
       'three_point_field_goals_attempted', 'total_rebounds',
       'total_technical_fouls', 'total_turnovers', 'turnover_points',
       'turnovers', 'opponent_team_id']]
    m_data = pd.merge(m_data, m_data, left_on=['game_id', 'opponent_team_id'], right_on=['game_id', 'team_id'], suffixes=('', '_opp'))

    basic_stats = m_data.groupby("team_id").agg(
        fg_attempted_sum = ("field_goals_attempted", "sum"),
        fg_made_sum = ("field_goals_made", "sum"),
        three_pt_made_sum=("three_point_field_goals_made", "sum"),
        three_pt_att_sum=("three_point_field_goals_attempted", "sum"),
        fta_sum=("free_throws_attempted", "sum"),
        ftm_sum=("free_throws_made", "sum"),
        to_sum=("turnovers", "sum"),
        orb_sum=("offensive_rebounds", "sum"),
        ast_sum = ("assists", "sum"),
        blk_sum = ("blocks", "sum"),
        stl_sum = ("steals", "sum"),
        fg_attempted_sum_opp = ("field_goals_attempted_opp", "sum")
    ).reset_index()

    team_stats = pd.merge(basic_stats, off_poss_count, on = 'team_id')
    team_stats = pd.merge(team_stats, transition_stats, on = 'team_id')

    new_cols = ['efgPct', 'tovPct', 'orbPct', 'ftaRate',
              'rimrate', 'rimFG', 'midrate', 'midfg',
              'fga3Rate', 'fg3Pct','ftPct', 'astPct',
              'stlPct', 'blkPct', 'transition_rate', 'transition_efg']

    team_stats[f'efgPct'] = (team_stats[f'fg_made_sum'] + 0.5 * team_stats[f'three_pt_made_sum']) / team_stats[f'fg_attempted_sum']
    team_stats[f'tovPct'] = team_stats[f'to_sum'] / team_stats[f'new_poss']
    team_stats[f'orbPct'] = team_stats[f'orb_sum'] / (team_stats[f'fg_attempted_sum'] - team_stats[f'fg_made_sum'])
    team_stats[f'ftaRate'] = team_stats[f'fta_sum'] / team_stats[f'fg_attempted_sum']
    team_stats[f'rimrate'] = team_stats[f'rim_fga'] / team_stats[f'fg_attempted_sum']
    team_stats[f'rimFG'] = team_stats[f'rim_fgm'] / team_stats[f'rim_fga']
    team_stats[f'midrate'] = team_stats[f'mid_fga'] / team_stats[f'fg_attempted_sum']
    team_stats[f'midfg'] = team_stats[f'mid_fgm'] / team_stats[f'mid_fga']
    team_stats[f'fga3Rate'] = team_stats[f'three_pt_att_sum'] / team_stats[f'fg_attempted_sum']
    team_stats[f'fg3Pct'] = team_stats[f'three_pt_made_sum'] / team_stats[f'three_pt_att_sum']

    team_stats[f'ftPct'] = team_stats[f'ftm_sum'] / team_stats[f'fta_sum']
    team_stats[f'astPct'] = team_stats[f'ast_sum'] / team_stats[f'fg_made_sum']
    team_stats[f'stlPct'] = team_stats[f'stl_sum'] / team_stats[f'new_poss_opp']
    team_stats[f'blkPct'] = team_stats['blk_sum'] / (team_stats['fg_attempted_sum_opp'])

    team_stats[f'transition_rate'] = (team_stats['2pa_transition'] + team_stats['3pa_transition']) / team_stats['fg_attempted_sum']
    team_stats[f'transition_efg'] = ((team_stats['2pm_transition'] + 1.5 * team_stats['3pm_transition']) / (team_stats['2pa_transition'] + team_stats['3pa_transition'])) / 100

    for col in new_cols:
        team_stats[f'{col}Pctile'] = team_stats[col].rank(pct=True)
    return team_stats.rename(columns={"team_id": "teamId"})

In [358]:
four_facts = build_fourfacts(2026)

100%|██████████| 1/1 [00:03<00:00,  3.40s/it]


In [343]:
import sportsdataverse as sdv
mbb_df = sdv.mbb.load_mbb_schedule(seasons=[2026])

100%|██████████| 1/1 [00:00<00:00,  1.90it/s]


In [365]:
import numpy as np
import pandas as pd

def agg_stats_sum(team_stats: pd.DataFrame) -> pd.Series:
    s = team_stats.sum(numeric_only=True)

    def safe_div(num, den):
        return np.nan if den == 0 else num / den

    out = {
        "efgPct": safe_div(s["fg_made_sum"] + 0.5 * s["three_pt_made_sum"], s["fg_attempted_sum"]),
        "tovPct": safe_div(s["to_sum"], s["new_poss"]),
        "orbPct": safe_div(s["orb_sum"], (s["fg_attempted_sum"] - s["fg_made_sum"])),
        "ftaRate": safe_div(s["fta_sum"], s["fg_attempted_sum"]),
        "rimrate": safe_div(s["rim_fga"], s["fg_attempted_sum"]),
        "rimFG": safe_div(s["rim_fgm"], s["rim_fga"]),
        "midrate": safe_div(s["mid_fga"], s["fg_attempted_sum"]),
        "midfg": safe_div(s["mid_fgm"], s["mid_fga"]),
        "fga3Rate": safe_div(s["three_pt_att_sum"], s["fg_attempted_sum"]),
        "fg3Pct": safe_div(s["three_pt_made_sum"], s["three_pt_att_sum"]),
        "ftPct": safe_div(s["ftm_sum"], s["fta_sum"]),
        "astPct": safe_div(s["ast_sum"], s["fg_made_sum"]),
        "stlPct": safe_div(s["stl_sum"], s["new_poss_opp"]),
        "blkPct": safe_div(s["blk_sum"], s["fg_attempted_sum_opp"]),
        "transition_rate": safe_div(s["2pa_transition"] + s["3pa_transition"], s["fg_attempted_sum"]),
        "transition_efg": safe_div(s["2pm_transition"] + 1.5 * s["3pm_transition"],
                                   s["2pa_transition"] + s["3pa_transition"]),
    }

    return pd.Series(out)


In [350]:
data = sdv.mbb.load_mbb_team_boxscore(seasons=[2026], return_as_pandas = True)

100%|██████████| 1/1 [00:00<00:00,  4.06it/s]


In [369]:
def agg_stats_sum(team_stats: pd.DataFrame) -> pd.Series:
    s = team_stats.sum(numeric_only=True)

    def safe_div(num, den):
        return np.nan if den == 0 else num / den

    out = {
        "efgPct": safe_div(s["fg_made_sum"] + 0.5 * s["three_pt_made_sum"], s["fg_attempted_sum"]),
        "tovPct": safe_div(s["to_sum"], s["new_poss"]),
        "orbPct": safe_div(s["orb_sum"], (s["fg_attempted_sum"] - s["fg_made_sum"])),
        "ftaRate": safe_div(s["fta_sum"], s["fg_attempted_sum"]),
        "rimrate": safe_div(s["rim_fga"], s["fg_attempted_sum"]),
        "rimFG": safe_div(s["rim_fgm"], s["rim_fga"]),
        "midrate": safe_div(s["mid_fga"], s["fg_attempted_sum"]),
        "midfg": safe_div(s["mid_fgm"], s["mid_fga"]),
        "fga3Rate": safe_div(s["three_pt_att_sum"], s["fg_attempted_sum"]),
        "fg3Pct": safe_div(s["three_pt_made_sum"], s["three_pt_att_sum"]),
        "ftPct": safe_div(s["ftm_sum"], s["fta_sum"]),
        "astPct": safe_div(s["ast_sum"], s["fg_made_sum"]),
        "stlPct": safe_div(s["stl_sum"], s["new_poss_opp"]),
        "blkPct": safe_div(s["blk_sum"], s["fg_attempted_sum_opp"]),
        "transition_rate": safe_div(s["2pa_transition"] + s["3pa_transition"], s["fg_attempted_sum"]),
        "transition_efg": safe_div(s["2pm_transition"] + 1.5 * s["3pm_transition"],
                                   s["2pa_transition"] + s["3pa_transition"]),
    }

    return pd.Series(out)


def build_opp_stats_df(four_facts: pd.DataFrame, year: int) -> pd.DataFrame:
    data = sdv.mbb.load_mbb_team_boxscore(seasons=[year], return_as_pandas=True)
    sched = (
        data[["team_id", "opponent_team_id"]]
        .dropna()
        .drop_duplicates()
    )

    opp_map = sched.groupby("team_id")["opponent_team_id"].apply(set).to_dict()

    rows = []
    for team_id, opps in opp_map.items():
        opp_facts = four_facts[four_facts["teamId"].isin(opps)]
        metrics = agg_stats_sum(opp_facts)  # Series of one-number-per-metric

        row = {"teamId": team_id}
        row.update(metrics.to_dict())
        rows.append(row)

    return pd.DataFrame(rows)


In [382]:
opp_sos_df = build_opp_stats_df(four_facts=four_facts.rename(columns={"team_id":"teamId"}), year=2026)

100%|██████████| 1/1 [00:00<00:00,  3.86it/s]


In [383]:
for col in opp_sos_df.columns[1:]:
    opp_sos_df[f'{col}Pctile'] = opp_sos_df[col].rank(pct=True)
opp_sos_df = opp_sos_df.add_prefix("opp_")
opp_sos_df.rename(columns={"opp_teamId": "teamId"}, inplace=True)

In [387]:
four_facts['transition_efg']

0        0.0075
1       0.00526
2      0.004691
3      0.005278
4      0.006443
         ...   
648    0.008571
649      0.0025
650    0.003421
651    0.002143
652      0.0025
Name: transition_efg, Length: 653, dtype: double[pyarrow]

In [381]:
opp_sos_df
sos_df = pd.merge(
    four_facts,
    opp_sos_df,
    on="teamId",
    suffixes=("", "_opp"),
)
sos_df

,teamId,fg_attempted_sum,fg_made_sum,three_pt_made_sum,three_pt_att_sum,fta_sum,ftm_sum,to_sum,orb_sum,ast_sum,...,opp_midratePctile,opp_midfgPctile,opp_fga3RatePctile,opp_fg3PctPctile,opp_ftPctPctile,opp_astPctPctile,opp_stlPctPctile,opp_blkPctPctile,opp_transition_ratePctile,opp_transition_efgPctile
0,1,56,23,8,28,10,7,14,9,9,...,0.934750,0.934750,0.479514,0.959029,0.962064,0.010622,0.016692,0.013657,0.009105,0.996965
1,2,611,290,80,236,273,199,108,141,134,...,0.493171,0.347496,0.681335,0.520486,0.704097,0.842185,0.688923,0.905918,0.403642,0.784522
2,5,666,296,57,212,249,175,100,156,145,...,0.834598,0.427162,0.236722,0.732929,0.467375,0.283763,0.854325,0.543247,0.567527,0.623672
3,6,572,268,69,195,228,151,86,101,125,...,0.781487,0.622155,0.229135,0.132018,0.205615,0.317147,0.738998,0.610015,0.679818,0.455235
4,8,570,270,78,220,217,170,85,98,154,...,0.250379,0.094082,0.905918,0.529590,0.394537,0.811836,0.623672,0.292868,0.918058,0.473445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
648,130262,56,27,7,16,28,16,36,5,20,...,0.600910,0.776935,0.933232,0.145675,0.458270,0.992413,0.940819,0.408194,0.779970,0.141123
649,131603,70,13,4,28,11,6,13,11,7,...,0.234446,0.990137,0.225341,0.917299,0.255690,0.170713,0.090288,0.242033,0.325493,0.888467
650,131632,68,23,9,38,10,5,11,7,9,...,0.005311,0.961305,0.955235,0.987102,0.990137,0.912747,0.358877,0.058422,0.947648,0.396813
651,131638,59,22,7,24,10,6,21,8,10,...,0.571320,0.899848,0.267830,0.091806,0.015175,0.692716,0.832322,0.238998,0.764036,0.462064
